[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/080.%20The%20International%20Freight%20Mode%20Selection%20Problem%20%28Air%20vs.%20Ocean%29/P80-Tier-5_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 80. The International Freight Mode Selection Problem (Air vs. Ocean)
## Tier 5: An Integrated Digital-Twin Simulation

### Purpose and scope

This notebook builds a small, simulated logistics digital twin to demonstrate how sensor readings and changing market conditions can feed a freight-mode decision process. It creates shipment and transport data, simulates IoT and market observations, estimates disruption risk, applies a rule-based air-versus-ocean assignment policy, and reports costs and performance indicators.

The decision policy follows the explicit rules in the code. It is **not** a mathematical optimizer: it does not search for a globally minimum-cost assignment, and its air-capacity figure is reported as a diagnostic rather than enforced as a constraint. The simulation is an educational prototype, not a live connection to physical logistics systems.

### Model assumptions

- The shipment list, base transportation costs, transit times, and capacity are illustrative model inputs.
- Market indices and sensor readings are synthetic. The random draws mean that repeated runs can produce different results.
- The market rate indices scale the corresponding mode's base transportation costs. Port congestion adds a surcharge to both modes and an integer delay to both modes. The fuel-price index is recorded and displayed but does not directly change costs in this decision policy.
- Predicted disruption delay is charged as an expected risk cost. This is separate from the scheduled-transit late penalty in the implemented cost formula; both terms are included as specified by the model.
- An early arrival incurs the model's holding cost for the number of days between arrival and the due date. A late arrival incurs the scheduled-transit late penalty.
- The assignment rules are applied shipment by shipment. They do not enforce aggregate air capacity, and a reported utilization above 100% indicates that assigned air weight exceeds the diagnostic capacity estimate.
- Sensor anomaly thresholds and market-risk calculations are simple illustrative rules, not calibrated predictive models. No claim of real-time synchronization, machine learning, or empirical forecasting is made.

### Simulation workflow

1. Define shipments, base transport parameters, and the simulated system inventory.
2. Generate synthetic sensor readings and market observations.
3. Summarize sensor data and estimate destination-level disruption risk.
4. Adjust costs and transit times for the observed market conditions.
5. Apply the stated assignment rules to each shipment and calculate its modeled cost.
6. Run baseline, named market scenarios, and what-if scenarios; compare their outputs.

### What to examine

- How the cost components combine for a shipment and how market conditions change them.
- How the priority and disruption rules determine an assignment.
- How scenario costs, air usage, risk exposure, alerts, and other indicators vary.
- Which conclusions follow from the model and which real-world effects the model leaves out.

## Decision policy and cost model

### Intuition

Each shipment is considered separately. The policy first gives high-priority shipments air freight when predicted disruption probability exceeds the stated threshold. Otherwise, it compares modeled ocean and air costs using a cost-advantage threshold, then checks a delay rule for high- or medium-priority shipments. If none of these conditions applies, it assigns ocean freight. The order matters: the first applicable rule determines the mode.

### Cost of one shipment under one mode

For shipment $i$ and mode $m$, the code calculates

$$
C_{im} = w_i c_{im} + H_{im} + L_{im} + R_{im},
$$

where

$$
H_{im} = \max(d_i - t_{im}, 0) r v_i,
\qquad
L_{im} = \max(t_{im} - d_i, 0) p,
\qquad
R_{im} = q_i \ell_i p.
$$

Here, $w_i$ is shipment weight in kilograms; $c_{im}$ is the market-adjusted transport cost per kilogram; $v_i$ is shipment value in dollars; $d_i$ is its due date in days; $t_{im}$ is the market-adjusted scheduled transit time in days; $r$ is the daily holding-cost rate; $p$ is the late-penalty amount per day; $q_i$ is the destination's predicted disruption probability; and $\ell_i$ is its predicted disruption delay in days. Thus, the four terms are transport, early-arrival holding, scheduled-transit lateness, and expected disruption cost, respectively. The corresponding code is `DigitalTwinAnalytics._calculate_adjusted_cost`.

### Assignment rule

Let $C_{i,o}$ and $C_{i,a}$ denote the modeled costs for ocean and air, respectively. The policy is:

$$
\text{mode}_i =
\begin{cases}
\text{air}, & \text{if priority}_i=\text{high and }q_i>0.5,\\
\text{ocean}, & \text{if }C_{i,o}<0.8C_{i,a},\\
\text{air}, & \text{if }\ell_i>3\text{ and priority}_i\in\{\text{high},\text{medium}\},\\
\text{ocean}, & \text{otherwise}.
\end{cases}
$$

The inequalities and their order are part of the policy. In particular, an air assignment is not necessarily the least-cost choice: the priority and delay rules can override a simple cost comparison. The program reports the sum of the costs for the selected assignments, but it does not solve a joint optimization problem. The decision implementation is `DigitalTwinAnalytics.optimize_freight_modes`.

### Risk and market adjustments

For each destination $j$, disruption probability is calculated as

$$
q_j = \min\left(0.95,\;0.6g_j+0.4u_j\right),
$$

where $g_j$ is the simulated port-congestion index and $u_j$ is the simulated weather-risk index. The code then draws a delay from a range selected by $q_j$: 3–10 days when $q_j>0.7$, 1–5 days when $0.4<q_j\leq0.7$, and zero otherwise. This is a simple stochastic rule, not a statistical forecast fitted to historical data.

The market rate index multiplies each mode's base cost per kilogram. Congestion adds a proportional surcharge and an integer transit-time delay. The adjusted air capacity is used to calculate a utilization indicator; it is not a constraint in the assignment rule. These calculations appear in `DigitalTwinAnalytics._adjust_parameters_for_market` and `DigitalTwinAnalytics._generate_disruption_predictions`.

In [ ]:
import random
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

## 1. Define the shipments and simulated environment

A shipment contains the attributes used by the cost model and decision policy. Transport costs are expressed in dollars per kilogram, transit times and due dates in days, shipment weights in kilograms, and shipment values in dollars. The digital-twin component counts describe the scale of the simulated infrastructure; they are not individual records of real assets.

In [ ]:
@dataclass
class Shipment:
    """Shipment attributes used by the mode-selection policy."""

    id: int
    weight: float  # kg
    value: float  # USD
    destination: str
    due_date: int  # days from the planning date
    priority: str  # high, medium, or low


@dataclass
class TransportParams:
    """Transportation costs, transit times, and cost-model parameters."""

    costs: Dict[str, Dict[str, float]]  # mode -> destination -> USD per kg
    transit_times: Dict[str, Dict[str, int]]  # mode -> destination -> days
    holding_rate: float  # daily holding cost as a fraction of shipment value
    late_penalty: float  # USD per late day
    air_capacity: float  # kg; used as a diagnostic capacity estimate


@dataclass
class IoTSensor:
    """A simulated sensor reading with its location, time, value, and unit."""

    sensor_id: str
    sensor_type: str
    location: str
    timestamp: datetime
    value: float
    unit: str


@dataclass
class MarketData:
    """A synthetic market observation for one simulation step."""

    timestamp: datetime
    ocean_rate_index: float
    air_rate_index: float
    fuel_price_index: float
    port_congestion_index: Dict[str, float]
    weather_disruption_risk: Dict[str, float]


def create_digital_twin_environment() -> Tuple[List[Shipment], TransportParams, Dict]:
    """Create the illustrative shipment set, transport inputs, and system inventory."""
    shipments = [
        Shipment(1, 800, 1_500_000, "rotterdam", 8, "high"),
        Shipment(2, 1_200, 800_000, "los_angeles", 15, "medium"),
        Shipment(3, 3_500, 600_000, "rotterdam", 25, "low"),
        Shipment(4, 600, 2_000_000, "singapore", 10, "high"),
        Shipment(5, 2_000, 400_000, "hamburg", 20, "medium"),
        Shipment(6, 1_500, 900_000, "new_york", 12, "high"),
        Shipment(7, 4_000, 300_000, "rotterdam", 35, "low"),
        Shipment(8, 900, 1_200_000, "los_angeles", 18, "medium"),
    ]

    base_params = TransportParams(
        costs={
            "ocean": {
                "rotterdam": 2.2,
                "los_angeles": 1.9,
                "singapore": 2.8,
                "hamburg": 2.1,
                "new_york": 1.8,
            },
            "air": {
                "rotterdam": 6.8,
                "los_angeles": 6.2,
                "singapore": 7.5,
                "hamburg": 6.5,
                "new_york": 5.9,
            },
        },
        transit_times={
            "ocean": {
                "rotterdam": 28,
                "los_angeles": 20,
                "singapore": 35,
                "hamburg": 26,
                "new_york": 18,
            },
            "air": {
                "rotterdam": 3,
                "los_angeles": 2,
                "singapore": 4,
                "hamburg": 3,
                "new_york": 2,
            },
        },
        holding_rate=0.0012,
        late_penalty=1500,
        air_capacity=15000,
    )

    twin_components = {
        "physical_assets": {
            "vessels": 25,
            "ports": 8,
            "containers": 5000,
            "warehouses": 12,
        },
        "connectivity": {
            "iot_sensors": 500,
            "rfid_readers": 50,
            "satellite_links": 15,
            "data_sources": 47,
        },
        "data_processing": {
            "update_frequency": 60,  # seconds
            "data_points_per_day": 2_300_000,
            "analytics_latency": 30,  # seconds
            "prediction_horizon": 7,  # days
        },
    }
    return shipments, base_params, twin_components


shipments, base_params, twin_components = create_digital_twin_environment()

print("Digital-twin simulation environment initialized")
print(f"  Shipments: {len(shipments)}")
print(f"  Physical assets: {twin_components['physical_assets']}")
print(f"  Connectivity: {twin_components['connectivity']}")
print(f"  Data processing: {twin_components['data_processing']}")

## 2. Simulate sensor and market feeds

The sensors and market feed are synthetic data generators. Their purpose is to provide changing inputs to the same analysis and assignment sequence used throughout the notebook. They do not connect to external services or physical devices. Each sensor object stores a timestamp and unit so that readings can be grouped and interpreted consistently.

In [ ]:
class IoTDataSimulator:
    """Generate synthetic readings for a simulated sensor network."""

    def __init__(self, twin_components: Dict):
        self.twin_components = twin_components
        self.sensor_types = [
            "position",
            "temperature",
            "humidity",
            "weight",
            "vibration",
            "pressure",
        ]
        self.locations = [
            "rotterdam",
            "los_angeles",
            "singapore",
            "hamburg",
            "new_york",
            "vessel_1",
            "vessel_2",
            "warehouse_1",
            "port_rotterdam",
            "port_la",
        ]
        self.current_time = datetime.now()
        self.sensors = self._initialize_sensors()

    def _initialize_sensors(self) -> List[IoTSensor]:
        """Distribute the configured sensor count across location/type pairs."""
        sensor_count = self.twin_components["connectivity"]["iot_sensors"]
        pairs = [
            (location, sensor_type)
            for location in self.locations
            for sensor_type in self.sensor_types
        ]
        sensors = []

        # Distributing the remainder ensures the simulated network matches its configured count.
        quotient, remainder = divmod(sensor_count, len(pairs))
        for pair_index, (location, sensor_type) in enumerate(pairs):
            count = quotient + (1 if pair_index < remainder else 0)
            for sensor_index in range(count):
                sensors.append(
                    IoTSensor(
                        sensor_id=f"{location}_{sensor_type}_{sensor_index}",
                        sensor_type=sensor_type,
                        location=location,
                        timestamp=self.current_time,
                        value=self._generate_sensor_value(sensor_type),
                        unit=self._get_sensor_unit(sensor_type),
                    )
                )
        return sensors

    def _generate_sensor_value(self, sensor_type: str) -> float:
        """Draw an illustrative value from a range appropriate to its sensor type."""
        ranges = {
            "position": (-180, 180),
            "temperature": (15, 35),
            "humidity": (30, 80),
            "weight": (100, 50_000),
            "vibration": (0, 10),
            "pressure": (980, 1050),
        }
        low, high = ranges.get(sensor_type, (0, 100))
        return random.uniform(low, high)

    def _get_sensor_unit(self, sensor_type: str) -> str:
        """Return the unit associated with a sensor type."""
        units = {
            "position": "degrees longitude",
            "temperature": "degrees Celsius",
            "humidity": "%",
            "weight": "kg",
            "vibration": "Hz",
            "pressure": "hPa",
        }
        return units.get(sensor_type, "unit")

    def generate_real_time_data(self, num_readings: int = 100) -> List[IoTSensor]:
        """Generate a batch of synthetic readings with small random variation."""
        if num_readings < 0:
            raise ValueError("num_readings must be non-negative")
        if num_readings and not self.sensors:
            raise ValueError("The simulated sensor network is empty")

        readings = []
        for _ in range(num_readings):
            sensor = random.choice(self.sensors)
            variation = random.gauss(0, abs(sensor.value) * 0.05)
            new_value = sensor.value + variation
            if sensor.sensor_type == "position":
                new_value = max(-180, min(180, new_value))
            else:
                new_value = max(0, new_value)

            readings.append(
                IoTSensor(
                    sensor_id=sensor.sensor_id,
                    sensor_type=sensor.sensor_type,
                    location=sensor.location,
                    timestamp=self.current_time,
                    value=new_value,
                    unit=sensor.unit,
                )
            )
            self.current_time += timedelta(seconds=random.uniform(1, 10))
        return readings


class MarketDataFeed:
    """Generate a synthetic sequence of market observations."""

    def __init__(self):
        self.current_time = datetime.now()
        self.destinations = [
            "rotterdam",
            "los_angeles",
            "singapore",
            "hamburg",
            "new_york",
        ]
        self.base_conditions = {
            "ocean_rate_index": 1.0,
            "air_rate_index": 1.0,
            "fuel_price_index": 1.0,
            "port_congestion": {destination: 0.5 for destination in self.destinations},
            "weather_risk": {destination: 0.1 for destination in self.destinations},
        }

    def generate_market_data(self) -> MarketData:
        """Update market conditions with bounded congestion and risk indices."""
        ocean_rate = max(
            0.5,
            self.base_conditions["ocean_rate_index"] * (1 + random.gauss(0, 0.02)),
        )
        air_rate = max(
            0.5,
            self.base_conditions["air_rate_index"] * (1 + random.gauss(0, 0.015)),
        )
        fuel_price = max(
            0.5,
            self.base_conditions["fuel_price_index"] * (1 + random.gauss(0, 0.025)),
        )

        congestion = {}
        weather_risk = {}
        for destination in self.destinations:
            congestion[destination] = max(
                0,
                min(
                    1,
                    self.base_conditions["port_congestion"][destination]
                    + random.gauss(0, 0.05),
                ),
            )
            weather_risk[destination] = max(
                0,
                min(
                    1,
                    self.base_conditions["weather_risk"][destination]
                    + random.gauss(0, 0.03),
                ),
            )

        market_data = MarketData(
            timestamp=self.current_time,
            ocean_rate_index=ocean_rate,
            air_rate_index=air_rate,
            fuel_price_index=fuel_price,
            port_congestion_index=congestion,
            weather_disruption_risk=weather_risk,
        )
        self.base_conditions.update(
            {
                "ocean_rate_index": ocean_rate,
                "air_rate_index": air_rate,
                "fuel_price_index": fuel_price,
                "port_congestion": congestion,
                "weather_risk": weather_risk,
            }
        )
        self.current_time += timedelta(minutes=random.uniform(5, 15))
        return market_data


iot_simulator = IoTDataSimulator(twin_components)
market_feed = MarketDataFeed()
print("Synthetic data simulators initialized")
print(f"  Configured IoT sensors: {len(iot_simulator.sensors)}")
print(f"  Market destinations: {len(market_feed.destinations)}")

## 3. Analyze sensor and market observations

The analytics component keeps a record of the observations it processes, summarizes sensor readings, applies fixed anomaly thresholds, and calculates destination-level disruption indicators. An anomaly is a reading outside the configured threshold for its sensor type; it is not proof of equipment failure. Market recommendations are descriptive rules and do not change the freight assignment policy unless the market values enter that policy through adjusted costs or risk.

In [ ]:
class DigitalTwinAnalytics:
    """Summarize synthetic inputs and apply the freight-mode decision policy."""

    def __init__(self, shipments: List[Shipment], params: TransportParams):
        self.shipments = shipments
        self.params = params
        self.historical_data = {
            "sensor_readings": [],
            "market_data": [],
            "decisions": [],
            "outcomes": [],
        }
        self.disruption_predictions = {}
        self.cost_forecasts = {}
        self.capacity_alerts = []
        self.kpis = {
            "on_time_delivery": [],
            "cost_efficiency": [],
            "capacity_utilization": [],
            "disruption_response_time": [],
            "prediction_accuracy": [],
        }

    def process_sensor_data(self, sensor_readings: List[IoTSensor]) -> Dict:
        """Group readings, identify threshold anomalies, and create alerts."""
        self.historical_data["sensor_readings"].extend(sensor_readings)
        analysis = {
            "total_readings": len(sensor_readings),
            "readings_by_type": {},
            "readings_by_location": {},
            "anomalies": [],
            "alerts": [],
        }

        for reading in sensor_readings:
            analysis["readings_by_type"].setdefault(reading.sensor_type, []).append(
                reading.value
            )
            analysis["readings_by_location"].setdefault(reading.location, []).append(
                reading.value
            )
            if self._detect_anomaly(reading):
                analysis["anomalies"].append(
                    {
                        "sensor_id": reading.sensor_id,
                        "type": reading.sensor_type,
                        "location": reading.location,
                        "value": reading.value,
                        "severity": "high" if abs(reading.value) > 1000 else "medium",
                    }
                )

        analysis["alerts"] = self._generate_alerts(analysis["anomalies"])
        return analysis

    def _detect_anomaly(self, reading: IoTSensor) -> bool:
        """Check a reading against illustrative sensor-specific limits."""
        thresholds = {
            "position": (-180, 180),
            "temperature": (-10, 50),
            "humidity": (0, 100),
            "vibration": (0, 20),
            "pressure": (900, 1100),
            "weight": (0, 100_000),
        }
        if reading.sensor_type not in thresholds:
            return False
        minimum, maximum = thresholds[reading.sensor_type]
        return reading.value < minimum or reading.value > maximum

    def _generate_alerts(self, anomalies: List[Dict]) -> List[Dict]:
        """Translate detected anomalies into descriptive monitoring alerts."""
        alerts = []
        for anomaly in anomalies:
            is_critical = anomaly["severity"] == "high"
            alerts.append(
                {
                    "type": "critical_anomaly" if is_critical else "warning",
                    "message": (
                        f"{'Critical anomaly' if is_critical else 'Anomalous reading'} "
                        f"in {anomaly['type']} sensor at {anomaly['location']}"
                    ),
                    "sensor_id": anomaly["sensor_id"],
                    "timestamp": datetime.now(),
                    "action_required": (
                        "immediate_investigation" if is_critical else "monitor"
                    ),
                }
            )
        return alerts

    def process_market_data(self, market_data: MarketData) -> Dict:
        """Summarize a market observation and calculate its risk indicators."""
        self.historical_data["market_data"].append(market_data)
        congestion_values = list(market_data.port_congestion_index.values())
        weather_values = list(market_data.weather_disruption_risk.values())
        return {
            "current_conditions": {
                "ocean_rate_trend": market_data.ocean_rate_index,
                "air_rate_trend": market_data.air_rate_index,
                "fuel_price_trend": market_data.fuel_price_index,
                "avg_congestion": float(np.mean(congestion_values)),
                "avg_weather_risk": float(np.mean(weather_values)),
            },
            "predictions": self._generate_disruption_predictions(market_data),
            "recommendations": self._generate_market_recommendations(market_data),
        }

    def _generate_disruption_predictions(self, market_data: MarketData) -> Dict:
        """Apply the stated congestion/weather rule and draw a delay category."""
        predictions = {}
        for destination, congestion in market_data.port_congestion_index.items():
            weather_risk = market_data.weather_disruption_risk[destination]
            probability = min(0.95, congestion * 0.6 + weather_risk * 0.4)

            if probability > 0.7:
                predicted_delay = random.randint(3, 10)
            elif probability > 0.4:
                predicted_delay = random.randint(1, 5)
            else:
                predicted_delay = 0

            predictions[destination] = {
                "disruption_probability": probability,
                "predicted_delay_days": predicted_delay,
                "confidence": min(0.9, probability * 1.2),
                "recommended_action": self._get_recommended_action(
                    probability, predicted_delay
                ),
            }
        self.disruption_predictions = predictions
        return predictions

    def _get_recommended_action(
        self, disruption_prob: float, predicted_delay: int
    ) -> str:
        """Map a risk estimate and delay draw to a descriptive action label."""
        if disruption_prob > 0.8 and predicted_delay > 5:
            return "immediate_air_diversion"
        if disruption_prob > 0.6 and predicted_delay > 3:
            return "partial_air_diversion"
        if disruption_prob > 0.4:
            return "increase_buffer_inventory"
        return "monitor_closely"

    def _generate_market_recommendations(self, market_data: MarketData) -> List[str]:
        """Create simple market-monitoring recommendations."""
        recommendations = []
        if market_data.ocean_rate_index > 1.3:
            recommendations.append(
                "Ocean rates are elevated; review time-sensitive freight plans."
            )
        elif market_data.ocean_rate_index < 0.8:
            recommendations.append(
                "Ocean rates are favorable; review ocean freight utilization."
            )
        if market_data.air_rate_index > 1.2:
            recommendations.append(
                "Air rates are elevated; review the use of air freight."
            )
        if market_data.fuel_price_index > 1.25:
            recommendations.append(
                "Fuel prices are elevated; review cost assumptions."
            )
        average_congestion = np.mean(
            list(market_data.port_congestion_index.values())
        )
        if average_congestion > 0.7:
            recommendations.append(
                "Port congestion is high; review capacity and routing plans."
            )
        return recommendations

    def optimize_freight_modes(self, market_data: MarketData) -> Dict:
        """Apply the shipment-by-shipment assignment rules and sum their costs."""
        predictions = self._generate_disruption_predictions(market_data)
        adjusted_params = self._adjust_parameters_for_market(market_data)
        results = {
            "assignments": {},
            "total_cost": 0.0,
            "risk_mitigation": {},
            "alternative_plans": {},
        }

        for shipment in self.shipments:
            risk = predictions.get(
                shipment.destination,
                {
                    "disruption_probability": 0.1,
                    "predicted_delay_days": 0,
                    "recommended_action": "monitor_closely",
                },
            )
            ocean_cost = self._calculate_adjusted_cost(
                shipment, "ocean", adjusted_params, risk
            )
            air_cost = self._calculate_adjusted_cost(
                shipment, "air", adjusted_params, risk
            )

            if (
                shipment.priority == "high"
                and risk["disruption_probability"] > 0.5
            ):
                assigned_mode = "air"
            elif ocean_cost < air_cost * 0.8:
                assigned_mode = "ocean"
            elif (
                risk["predicted_delay_days"] > 3
                and shipment.priority in ["high", "medium"]
            ):
                assigned_mode = "air"
            else:
                assigned_mode = "ocean"

            final_cost = self._calculate_adjusted_cost(
                shipment, assigned_mode, adjusted_params, risk
            )
            results["assignments"][shipment.id] = {
                "mode": assigned_mode,
                "cost": final_cost,
                "risk_adjusted_cost": final_cost,
                "disruption_probability": risk["disruption_probability"],
                "recommended_action": risk["recommended_action"],
            }
            results["total_cost"] += final_cost

        self.historical_data["decisions"].append(results["assignments"])
        return results

    def _adjust_parameters_for_market(
        self, market_data: MarketData
    ) -> TransportParams:
        """Apply rate indices and congestion adjustments to the base parameters."""
        adjusted_costs = {}
        adjusted_transit_times = {}
        for mode in ["ocean", "air"]:
            rate_index = (
                market_data.ocean_rate_index
                if mode == "ocean"
                else market_data.air_rate_index
            )
            adjusted_costs[mode] = {}
            adjusted_transit_times[mode] = {}
            for destination in self.params.costs[mode]:
                congestion = market_data.port_congestion_index[destination]
                surcharge = congestion * 0.2
                adjusted_costs[mode][destination] = (
                    self.params.costs[mode][destination]
                    * rate_index
                    * (1 + surcharge)
                )
                congestion_delay = int(congestion * 3)
                adjusted_transit_times[mode][destination] = (
                    self.params.transit_times[mode][destination] + congestion_delay
                )

        mean_congestion = np.mean(
            list(market_data.port_congestion_index.values())
        )
        adjusted_air_capacity = self.params.air_capacity * (
            1 - mean_congestion * 0.3
        )
        return TransportParams(
            costs=adjusted_costs,
            transit_times=adjusted_transit_times,
            holding_rate=self.params.holding_rate,
            late_penalty=self.params.late_penalty,
            air_capacity=adjusted_air_capacity,
        )

    def _calculate_adjusted_cost(
        self,
        shipment: Shipment,
        mode: str,
        params: TransportParams,
        risk: Dict,
    ) -> float:
        """Calculate transport, holding, lateness, and expected disruption costs."""
        transport_cost = shipment.weight * params.costs[mode][shipment.destination]
        transit_time = params.transit_times[mode][shipment.destination]
        expected_delay_cost = (
            risk["predicted_delay_days"]
            * params.late_penalty
            * risk["disruption_probability"]
        )

        early_days = max(shipment.due_date - transit_time, 0)
        holding_cost = early_days * params.holding_rate * shipment.value
        late_days = max(transit_time - shipment.due_date, 0)
        late_penalty_cost = late_days * params.late_penalty

        return (
            transport_cost
            + holding_cost
            + late_penalty_cost
            + expected_delay_cost
        )


analytics = DigitalTwinAnalytics(shipments, base_params)
print("Analytics component initialized")
print("  Sensor summaries, market-risk indicators, and mode assignments are available")

## 4. Integrate the components in a simulation step

A simulation step follows the same sequence each time: generate sensor readings, summarize them, generate a market observation, analyze it, apply the mode-assignment policy, and calculate indicators. This ordering makes the path from input data to reported decision explicit. A scenario changes the feed's initial conditions and then runs this same sequence repeatedly.

In [ ]:
class DigitalTwinSimulation:
    """Coordinate synthetic feeds, analytics, assignments, and reporting."""

    def __init__(
        self,
        shipments: List[Shipment],
        params: TransportParams,
        twin_components: Dict,
    ):
        self.shipments = shipments
        self.params = params
        self.twin_components = twin_components
        self.iot_simulator = IoTDataSimulator(twin_components)
        self.market_feed = MarketDataFeed()
        self.analytics = DigitalTwinAnalytics(shipments, params)
        self.current_time = datetime.now()
        self.simulation_step = 0
        self.simulation_results = {
            "sensor_analytics": [],
            "market_analytics": [],
            "optimization_results": [],
            "alerts": [],
            "kpis": [],
            "scenario_analysis": {},
        }

    def run_simulation_step(self) -> Dict:
        """Run one complete sensor-to-assignment simulation cycle."""
        step_results = {
            "timestamp": self.current_time,
            "step": self.simulation_step,
            "sensor_data": None,
            "market_data": None,
            "optimization": None,
            "alerts": [],
            "kpis": {},
        }

        sensor_readings = self.iot_simulator.generate_real_time_data(50)
        sensor_analysis = self.analytics.process_sensor_data(sensor_readings)
        step_results["sensor_data"] = sensor_analysis

        market_data = self.market_feed.generate_market_data()
        market_analysis = self.analytics.process_market_data(market_data)
        step_results["market_data"] = market_analysis

        optimization_results = self.analytics.optimize_freight_modes(market_data)
        step_results["optimization"] = optimization_results
        step_results["alerts"] = sensor_analysis["alerts"]
        step_results["kpis"] = self._calculate_step_kpis(step_results)

        self.simulation_results["sensor_analytics"].append(sensor_analysis)
        self.simulation_results["market_analytics"].append(market_analysis)
        self.simulation_results["optimization_results"].append(optimization_results)
        self.simulation_results["alerts"].extend(step_results["alerts"])
        self.simulation_results["kpis"].append(step_results["kpis"])

        self.current_time += timedelta(minutes=30)
        self.simulation_step += 1
        return step_results

    def _calculate_step_kpis(self, step_results: Dict) -> Dict:
        """Calculate cost, air-capacity, risk, sensor, and alert indicators."""
        kpis = {}
        optimization = step_results["optimization"]
        if optimization:
            total_value = sum(shipment.value for shipment in self.shipments)
            kpis["cost_efficiency"] = (
                optimization["total_cost"] / total_value if total_value > 0 else 0
            )
            assignments = optimization["assignments"]
            air_count = sum(
                assignment["mode"] == "air" for assignment in assignments.values()
            )
            air_weight = sum(
                shipment.weight
                for shipment in self.shipments
                if assignments[shipment.id]["mode"] == "air"
            )
            average_congestion = step_results["market_data"][
                "current_conditions"
            ]["avg_congestion"]
            current_capacity = self.params.air_capacity * (
                1 - average_congestion * 0.3
            )
            kpis["air_capacity_utilization"] = (
                air_weight / current_capacity if current_capacity > 0 else 0
            )
            kpis["air_shipment_percentage"] = (
                air_count / len(self.shipments) if self.shipments else 0
            )
            kpis["average_risk_exposure"] = (
                np.mean(
                    [
                        assignment["disruption_probability"]
                        for assignment in assignments.values()
                    ]
                )
                if assignments
                else 0
            )

        sensor_data = step_results["sensor_data"]
        if sensor_data:
            total_readings = sensor_data["total_readings"]
            kpis["sensor_data_volume"] = total_readings
            kpis["anomaly_rate"] = (
                len(sensor_data["anomalies"]) / total_readings
                if total_readings
                else 0
            )

        alerts = step_results["alerts"]
        kpis["critical_alerts"] = sum(
            alert["type"] == "critical_anomaly" for alert in alerts
        )
        kpis["warning_alerts"] = sum(
            alert["type"] == "warning" for alert in alerts
        )
        return kpis

    def _configure_scenario(self, scenario_name: str) -> None:
        """Set the market feed's starting conditions for a named scenario."""
        destinations = self.market_feed.destinations
        if scenario_name == "hurricane_season":
            self.market_feed.base_conditions.update(
                {
                    "ocean_rate_index": 1.4,
                    "air_rate_index": 1.2,
                    "weather_risk": {destination: 0.8 for destination in destinations},
                }
            )
        elif scenario_name == "fuel_crisis":
            self.market_feed.base_conditions.update(
                {
                    "fuel_price_index": 1.8,
                    "ocean_rate_index": 1.3,
                    "air_rate_index": 1.6,
                }
            )
        elif scenario_name == "port_congestion":
            self.market_feed.base_conditions.update(
                {
                    "port_congestion": {
                        destination: 0.9 for destination in destinations
                    },
                    "air_rate_index": 1.3,
                }
            )
        else:
            raise ValueError(f"Unknown scenario: {scenario_name}")

    def run_scenario_analysis(self, scenario_name: str, num_steps: int = 20) -> Dict:
        """Run a scenario from common baseline conditions and summarize its steps."""
        if num_steps <= 0:
            raise ValueError("num_steps must be positive")

        print(f"\nRunning {scenario_name.replace('_', ' ').title()} scenario...")
        baseline_conditions = {
            "ocean_rate_index": 1.0,
            "air_rate_index": 1.0,
            "fuel_price_index": 1.0,
            "port_congestion": {
                destination: 0.5 for destination in self.market_feed.destinations
            },
            "weather_risk": {
                destination: 0.1 for destination in self.market_feed.destinations
            },
        }
        self.market_feed.base_conditions = {
            key: value.copy() if isinstance(value, dict) else value
            for key, value in baseline_conditions.items()
        }
        self._configure_scenario(scenario_name)

        scenario_results = []
        for step in range(num_steps):
            step_result = self.run_simulation_step()
            scenario_results.append(step_result)
            if step % 5 == 0:
                print(
                    f"  Step {step}: "
                    f"Cost=${step_result['optimization']['total_cost']:,.0f}, "
                    f"Alerts={len(step_result['alerts'])}"
                )

        cost_evolution = [
            result["optimization"]["total_cost"] for result in scenario_results
        ]
        scenario_alerts = sum(
            len(result["alerts"]) for result in scenario_results
        )
        analysis = {
            "scenario_name": scenario_name,
            "total_steps": num_steps,
            "final_cost": cost_evolution[-1],
            "cost_evolution": cost_evolution,
            "cost_volatility": float(np.std(cost_evolution)),
            "alert_frequency": scenario_alerts / num_steps,
            "final_assignments": scenario_results[-1]["optimization"]["assignments"],
            "kpis_trend": {
                name: [result["kpis"].get(name, 0) for result in scenario_results]
                for name in [
                    "cost_efficiency",
                    "air_capacity_utilization",
                    "average_risk_exposure",
                ]
            },
        }
        self.simulation_results["scenario_analysis"][scenario_name] = analysis
        return analysis

    def generate_comprehensive_report(self) -> Dict:
        """Summarize accumulated simulation steps and scenario comparisons."""
        kpis = self.simulation_results["kpis"]
        mean_kpi = lambda name: float(
            np.mean([item.get(name, 0) for item in kpis])
        ) if kpis else 0.0
        return {
            "summary": {
                "simulation_steps": self.simulation_step,
                "total_data_points": sum(
                    item["total_readings"]
                    for item in self.simulation_results["sensor_analytics"]
                ),
                "total_alerts": len(self.simulation_results["alerts"]),
                "active_scenarios": list(
                    self.simulation_results["scenario_analysis"].keys()
                ),
            },
            "performance_metrics": {
                "avg_cost_efficiency": mean_kpi("cost_efficiency"),
                "avg_air_utilization": mean_kpi("air_capacity_utilization"),
                "avg_risk_exposure": mean_kpi("average_risk_exposure"),
                "data_quality_score": self._calculate_data_quality_score(),
            },
            "scenario_comparisons": self._compare_scenarios(),
            "recommendations": self._generate_recommendations(),
        }

    def _calculate_data_quality_score(self) -> float:
        """Return a simple score based on the observed sensor anomaly rate."""
        analyses = self.simulation_results["sensor_analytics"]
        total_readings = sum(item["total_readings"] for item in analyses)
        if total_readings == 0:
            return 0.0
        total_anomalies = sum(len(item["anomalies"]) for item in analyses)
        return max(0.0, 1 - (total_anomalies / total_readings) * 10)

    def _compare_scenarios(self) -> Dict:
        """Collect comparable cost, alert, and risk summaries by scenario."""
        comparisons = {}
        for name, analysis in self.simulation_results["scenario_analysis"].items():
            risk_values = analysis["kpis_trend"]["average_risk_exposure"]
            comparisons[name] = {
                "final_cost": analysis["final_cost"],
                "cost_volatility": analysis["cost_volatility"],
                "alert_frequency": analysis["alert_frequency"],
                "risk_management": float(np.mean(risk_values)) if risk_values else 0,
            }
        return comparisons

    def _generate_recommendations(self) -> List[str]:
        """Create threshold-based suggestions from the simulated indicators."""
        kpis = self.simulation_results["kpis"]
        if not kpis:
            return ["Run simulation steps before interpreting performance indicators."]

        recommendations = []
        average_cost_efficiency = np.mean(
            [item.get("cost_efficiency", 0) for item in kpis]
        )
        if average_cost_efficiency > 0.15:
            recommendations.append(
                "Modeled freight cost is high relative to cargo value; review cost assumptions."
            )

        average_air_utilization = np.mean(
            [item.get("air_capacity_utilization", 0) for item in kpis]
        )
        if average_air_utilization > 0.9:
            recommendations.append(
                "Diagnostic air-capacity utilization is high; review assigned air weight."
            )
        elif average_air_utilization < 0.3:
            recommendations.append(
                "Diagnostic air-capacity utilization is low; review mode assignments."
            )

        average_risk = np.mean(
            [item.get("average_risk_exposure", 0) for item in kpis]
        )
        if average_risk > 0.6:
            recommendations.append(
                "Average modeled risk is high; review the disruption assumptions."
            )

        if self._calculate_data_quality_score() < 0.8:
            recommendations.append(
                "The threshold-based data-quality score is low; inspect simulated anomalies."
            )
        if not recommendations:
            recommendations.append(
                "No recommendation thresholds were triggered by the simulated indicators."
            )
        return recommendations


digital_twin = DigitalTwinSimulation(shipments, base_params, twin_components)
print("Digital-twin simulation coordinator initialized")

## 5. Run the baseline and market scenarios

The baseline consists of repeated simulation steps under the feed's default conditions. Each named scenario starts from the same defined baseline market conditions before its scenario adjustments are applied; within a scenario, market conditions evolve from one step to the next. The random sensor and market draws are not seeded, so displayed results are run-specific rather than guaranteed constants.

In [ ]:
print("Starting the digital-twin simulation")
print("=" * 72)

baseline_steps = 10
for step in range(baseline_steps):
    step_result = digital_twin.run_simulation_step()
    if step % 3 == 0:
        print(
            f"Step {step}: "
            f"Cost=${step_result['optimization']['total_cost']:,.0f}, "
            f"Alerts={len(step_result['alerts'])}, "
            f"Air utilization="
            f"{step_result['kpis'].get('air_capacity_utilization', 0):.1%}"
        )
print(f"\nBaseline simulation completed: {baseline_steps} steps")

scenarios = ["hurricane_season", "fuel_crisis", "port_congestion"]
scenario_results = {
    name: digital_twin.run_scenario_analysis(name, num_steps=15)
    for name in scenarios
}

print("\nScenario analysis results")
for name, results in scenario_results.items():
    print(f"  {name.replace('_', ' ').title()}:")
    print(f"    Final cost: ${results['final_cost']:,.0f}")
    print(f"    Cost volatility: ${results['cost_volatility']:,.0f}")
    print(f"    Alerts per step: {results['alert_frequency']:.2f}")

comprehensive_report = digital_twin.generate_comprehensive_report()
print("\nSimulation report")
print("=" * 72)
print(f"Simulation steps: {comprehensive_report['summary']['simulation_steps']}")
print(f"Sensor readings: {comprehensive_report['summary']['total_data_points']:,}")
print(f"Total alerts: {comprehensive_report['summary']['total_alerts']}")
print(f"Scenarios analyzed: {len(comprehensive_report['summary']['active_scenarios'])}")

metrics = comprehensive_report["performance_metrics"]
print("\nPerformance indicators")
print(f"  Average cost-to-value ratio: {metrics['avg_cost_efficiency']:.3f}")
print(f"  Average diagnostic air utilization: {metrics['avg_air_utilization']:.1%}")
print(f"  Average modeled risk exposure: {metrics['avg_risk_exposure']:.1%}")
print(f"  Threshold-based data-quality score: {metrics['data_quality_score']:.1%}")

print("\nThreshold-based recommendations")
for index, recommendation in enumerate(comprehensive_report["recommendations"], 1):
    print(f"  {index}. {recommendation}")

### Interpreting the scenario indicators

`final_cost` is the total modeled cost of the assignments in the scenario's final simulation step. `cost_volatility` is the standard deviation of total modeled cost across that scenario's steps. `alert_frequency` is the number of alerts generated during that scenario divided by its number of steps; it does not include alerts from the baseline or other scenarios. Air-capacity utilization is a diagnostic ratio of assigned air weight to adjusted estimated capacity. Since the policy has no capacity constraint, a value above 100% is possible and should be interpreted as a capacity warning, not as a feasible capacity plan.

The fuel-crisis scenario changes the fuel-price index, but that index is not applied to transport costs by the current model. It can therefore affect reporting but does not, by itself, imply a cost change. Recommendations are threshold-based descriptions of these simulated indicators, not operational instructions.

## 6. Visualize scenario costs and assignments

The first chart compares the total modeled cost over the steps of each named scenario. The second uses separate vertical axes because cost, alerts per step, and cost volatility have different scales and units. The final panel shows the shipment assignments and costs from the last named scenario; color distinguishes air from ocean, and the legend explains that distinction. A marker identifies assignments whose modeled disruption probability exceeds the display threshold.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ax1, ax2, ax3, ax4 = axes.flat
fig.suptitle("Digital-Twin Freight Mode Simulation", fontsize=16, fontweight="bold")

for name, results in scenario_results.items():
    ax1.plot(
        range(len(results["cost_evolution"])),
        results["cost_evolution"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=name.replace("_", " ").title(),
    )
ax1.set_xlabel("Simulation step")
ax1.set_ylabel("Total modeled cost (USD)")
ax1.set_title("Cost by Scenario and Step")
ax1.legend()
ax1.grid(True, alpha=0.3)

scenario_names = list(scenario_results)
final_costs = [scenario_results[name]["final_cost"] for name in scenario_names]
volatilities = [scenario_results[name]["cost_volatility"] for name in scenario_names]
alert_rates = [scenario_results[name]["alert_frequency"] for name in scenario_names]
x = np.arange(len(scenario_names))
width = 0.25
ax2.bar(x - width, final_costs, width, color="steelblue", label="Final cost (USD)")
ax2.set_ylabel("Final modeled cost (USD)", color="steelblue")
ax2.tick_params(axis="y", labelcolor="steelblue")
ax2_twin = ax2.twinx()
ax2_twin.bar(x, alert_rates, width, color="darkorange", label="Alerts per step")
ax2_twin.bar(
    x + width,
    np.array(volatilities) / 1000,
    width,
    color="firebrick",
    label="Cost volatility (thousand USD)",
)
ax2_twin.set_ylabel("Alerts per step / cost volatility (thousand USD)")
ax2.set_title("Final Cost, Alerts, and Cost Volatility")
ax2.set_xlabel("Scenario")
ax2.set_xticks(x)
ax2.set_xticklabels([name.replace("_", "\n").title() for name in scenario_names])
ax2.legend(loc="upper left")
ax2_twin.legend(loc="upper right")
ax2.grid(True, alpha=0.3)

kpi_history = digital_twin.simulation_results["kpis"]
steps = range(len(kpi_history))
for key, label in [
    ("cost_efficiency", "Cost-to-value ratio"),
    ("air_capacity_utilization", "Diagnostic air utilization"),
    ("average_risk_exposure", "Average modeled risk"),
]:
    ax3.plot(steps, [item.get(key, 0) for item in kpi_history], label=label)
ax3.set_xlabel("Simulation step")
ax3.set_ylabel("Indicator value (ratio)")
ax3.set_title("Simulation Indicator Trends")
ax3.legend()
ax3.grid(True, alpha=0.3)

last_scenario = scenario_names[-1]
final_assignments = scenario_results[last_scenario]["final_assignments"]
shipment_ids = list(final_assignments)
modes = [final_assignments[item]["mode"] for item in shipment_ids]
costs = [final_assignments[item]["cost"] for item in shipment_ids]
risks = [
    final_assignments[item]["disruption_probability"] for item in shipment_ids
]
colors = ["lightcoral" if mode == "air" else "lightblue" for mode in modes]
bars = ax4.bar(shipment_ids, costs, color=colors, alpha=0.8)
for bar, mode, risk in zip(bars, modes, risks):
    height = bar.get_height()
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        height * 0.95,
        mode.upper(),
        ha="center",
        va="top",
        fontweight="bold",
    )
    if risk > 0.6:
        ax4.scatter(
            bar.get_x() + bar.get_width() / 2,
            height * 1.04,
            marker="x",
            color="black",
            label="Risk probability > 0.6" if bar is bars[0] else None,
        )
ax4.set_xlabel("Shipment ID")
ax4.set_ylabel("Modeled shipment cost (USD)")
ax4.set_title(f"Assignments: {last_scenario.replace('_', ' ').title()}")
ax4.set_xticks(shipment_ids)
ax4.grid(True, alpha=0.3)
mode_handles = [
    plt.Rectangle((0, 0), 1, 1, color="lightcoral", label="Air"),
    plt.Rectangle((0, 0), 1, 1, color="lightblue", label="Ocean"),
    plt.Line2D([], [], marker="x", linestyle="", color="black", label="Risk probability > 0.6"),
]
ax4.legend(handles=mode_handles)

plt.tight_layout()
plt.show()

print("Visualization guide")
print("  Cost curves show scenario-level variation across simulation steps.")
print("  The comparison panel uses separate axes for quantities with different units.")
print("  Indicator trends summarize recorded steps, including baseline and scenarios.")
print("  Assignment colors show the selected mode; x markers show elevated modeled risk.")

## 7. Run predictive and what-if scenarios

Each what-if case applies its specified market adjustments, then runs the same simulation-step sequence for 12 steps. The scenario feed starts from the common baseline conditions in each case, so a case does not inherit the prior case's market indices. The feed still evolves within each case. The adaptation indicator below compares average absolute cost changes in the early and later portions of one cost sequence; it is a descriptive statistic, not evidence that the policy learns or improves its decisions over time.

In [ ]:
def _calculate_adaptation_speed(cost_trend: List[float]) -> float:
    """Compare later and earlier mean absolute cost changes as a descriptive ratio."""
    if len(cost_trend) < 3:
        return 0.0
    changes = [
        abs(cost_trend[index + 1] - cost_trend[index])
        for index in range(len(cost_trend) - 1)
    ]
    midpoint = len(changes) // 2
    early_volatility = float(np.mean(changes[:midpoint]))
    late_volatility = float(np.mean(changes[midpoint:]))
    if early_volatility == 0:
        return 1.0 if late_volatility == 0 else 0.0
    return max(0.0, 1 - late_volatility / early_volatility)


def _generate_scenario_recommendations(scenario_results: List[Dict]) -> List[str]:
    """Generate simple threshold-based suggestions from one scenario's outputs."""
    if not scenario_results:
        return ["No scenario results are available for interpretation."]

    final_state = scenario_results[-1]
    assignments = final_state["optimization"]["assignments"]
    if not assignments:
        return ["No shipment assignments are available for interpretation."]

    recommendations = []
    air_share = sum(
        assignment["mode"] == "air" for assignment in assignments.values()
    ) / len(assignments)
    if air_share > 0.7:
        recommendations.append(
            "Air assignment share is high; review the diagnostic capacity estimate."
        )
    elif air_share < 0.2:
        recommendations.append(
            "Air assignment share is low; review the policy thresholds and cost inputs."
        )

    high_risk_count = sum(
        assignment["disruption_probability"] > 0.7
        for assignment in assignments.values()
    )
    if high_risk_count > len(assignments) * 0.5:
        recommendations.append(
            "Many assignments have high modeled risk; review disruption assumptions."
        )

    cost_trend = [
        result["optimization"]["total_cost"] for result in scenario_results
    ]
    if len(cost_trend) > 1 and cost_trend[0] != 0:
        cost_increase = (cost_trend[-1] - cost_trend[0]) / abs(cost_trend[0]) * 100
        if cost_increase > 20:
            recommendations.append(
                "Modeled cost increased substantially; review market and cost inputs."
            )
    if not recommendations:
        recommendations.append(
            "No what-if recommendation thresholds were triggered."
        )
    return recommendations


def demonstrate_predictive_capabilities(
    digital_twin: DigitalTwinSimulation,
) -> Dict:
    """Run the defined what-if cases using the existing simulation policy."""
    destinations = digital_twin.market_feed.destinations
    baseline_conditions = {
        "ocean_rate_index": 1.0,
        "air_rate_index": 1.0,
        "fuel_price_index": 1.0,
        "port_congestion": {destination: 0.5 for destination in destinations},
        "weather_risk": {destination: 0.1 for destination in destinations},
    }
    what_if_scenarios = {
        "red_sea_crisis": {
            "description": "Illustrative ocean-rate increase and elevated weather risk.",
            "market_adjustments": {
                "ocean_rate_index": 1.4,
                "air_rate_index": 1.15,
                "weather_risk": {destination: 0.5 for destination in destinations},
            },
        },
        "pandemic_recovery": {
            "description": "Illustrative elevated rates and port congestion.",
            "market_adjustments": {
                "ocean_rate_index": 1.25,
                "air_rate_index": 1.4,
                "port_congestion": {
                    destination: 0.85 for destination in destinations
                },
            },
        },
        "fuel_price_shock": {
            "description": "Illustrative fuel-index and freight-rate increases.",
            "market_adjustments": {
                "fuel_price_index": 2.0,
                "ocean_rate_index": 1.35,
                "air_rate_index": 1.6,
            },
        },
        "perfect_conditions": {
            "description": "Illustrative lower rates, congestion, and weather risk.",
            "market_adjustments": {
                "ocean_rate_index": 0.85,
                "air_rate_index": 0.9,
                "port_congestion": {
                    destination: 0.2 for destination in destinations
                },
                "weather_risk": {
                    destination: 0.05 for destination in destinations
                },
            },
        },
    }

    predictive_results = {}
    print("Running predictive and what-if scenarios")
    for name, configuration in what_if_scenarios.items():
        digital_twin.market_feed.base_conditions = {
            key: value.copy() if isinstance(value, dict) else value
            for key, value in baseline_conditions.items()
        }
        digital_twin.market_feed.base_conditions.update(
            configuration["market_adjustments"]
        )
        step_results = [
            digital_twin.run_simulation_step() for _ in range(12)
        ]
        final_state = step_results[-1]
        final_cost = final_state["optimization"]["total_cost"]
        cost_trend = [
            result["optimization"]["total_cost"] for result in step_results
        ]
        assignments = final_state["optimization"]["assignments"]
        air_percentage = (
            100
            * sum(item["mode"] == "air" for item in assignments.values())
            / len(digital_twin.shipments)
            if digital_twin.shipments
            else 0
        )
        average_risk = float(
            np.mean(
                [item["disruption_probability"] for item in assignments.values()]
            )
        ) if assignments else 0.0
        recommendations = _generate_scenario_recommendations(step_results)
        predictive_results[name] = {
            "description": configuration["description"],
            "final_cost": final_cost,
            "cost_trend": cost_trend,
            "air_usage_percentage": air_percentage,
            "average_risk": average_risk,
            "adaptation_speed": _calculate_adaptation_speed(cost_trend),
            "recommendations": recommendations,
        }
        print(f"\n{name.replace('_', ' ').title()}")
        print(f"  Final modeled cost: ${final_cost:,.0f}")
        print(f"  Air assignment share: {air_percentage:.1f}%")
        print(f"  Average modeled risk: {average_risk:.1%}")
        print(
            "  Cost-change stabilization indicator: "
            f"{predictive_results[name]['adaptation_speed']:.2f}"
        )

    digital_twin.market_feed.base_conditions = {
        key: value.copy() if isinstance(value, dict) else value
        for key, value in baseline_conditions.items()
    }
    return predictive_results


predictive_results = demonstrate_predictive_capabilities(digital_twin)
scenario_names = list(predictive_results)
final_costs = [predictive_results[name]["final_cost"] for name in scenario_names]
air_usage = [
    predictive_results[name]["air_usage_percentage"] for name in scenario_names
]
risk_levels = [
    predictive_results[name]["average_risk"] * 100 for name in scenario_names
]
adaptation_speeds = [
    predictive_results[name]["adaptation_speed"] for name in scenario_names
]

### What-if results and interpretation

The what-if cases are defined by the input adjustments in the code. Their results depend on random draws from the market and delay simulations. The adaptation indicator is bounded below by zero and compares observed cost variation; it does not measure decision quality, convergence to an optimum, or learning. In particular, the assignment policy is fixed throughout the simulation. Fuel-price changes alone do not alter modeled freight cost because fuel price is not used in the cost-adjustment equation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ax1, ax2, ax3, ax4 = axes.flat
fig.suptitle("What-If Scenario Analysis", fontsize=16, fontweight="bold")

x = np.arange(len(scenario_names))
width = 0.25
ax1.bar(x - width, final_costs, width, color="steelblue", label="Final cost (USD)")
ax1_twin = ax1.twinx()
ax1_twin.bar(x, air_usage, width, color="darkorange", label="Air assignment share (%)")
ax1_twin.bar(x + width, risk_levels, width, color="firebrick", label="Risk (%)")
ax1.set_xlabel("What-if scenario")
ax1.set_ylabel("Final modeled cost (USD)", color="steelblue")
ax1_twin.set_ylabel("Air assignment share / risk (%)")
ax1.set_title("Final Cost, Air Share, and Modeled Risk")
ax1.set_xticks(x)
ax1.set_xticklabels([name.replace("_", "\n").title() for name in scenario_names])
ax1.legend(loc="upper left")
ax1_twin.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

colors = [
    "green" if speed > 0.7 else "orange" if speed > 0.4 else "firebrick"
    for speed in adaptation_speeds
]
bars = ax2.bar(scenario_names, adaptation_speeds, color=colors, alpha=0.75)
ax2.set_xlabel("Scenario")
ax2.set_ylabel("Cost-change stabilization indicator")
ax2.set_title("Descriptive Cost-Change Indicator")
ax2.set_ylim(0, 1)
ax2.tick_params(axis="x", rotation=35)
ax2.grid(True, alpha=0.3)
ax2.axhline(0.7, color="green", linestyle="--", label="0.7 reference")
ax2.axhline(0.4, color="darkorange", linestyle="--", label="0.4 reference")
ax2.legend()
for bar, speed in zip(bars, adaptation_speeds):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        min(bar.get_height() + 0.02, 0.98),
        f"{speed:.2f}",
        ha="center",
        va="bottom",
    )

if "red_sea_crisis" in predictive_results:
    cost_trend = predictive_results["red_sea_crisis"]["cost_trend"]
    trend_steps = np.arange(len(cost_trend))
    ax3.plot(
        trend_steps,
        cost_trend,
        "o-",
        color="firebrick",
        linewidth=2,
        label="Modeled cost",
    )
    if len(cost_trend) >= 2:
        coefficients = np.polyfit(trend_steps, cost_trend, 1)
        trend_line = np.poly1d(coefficients)
        ax3.plot(
            trend_steps,
            trend_line(trend_steps),
            "--",
            color="black",
            alpha=0.6,
            label="Linear descriptive trend",
        )
    ax3.set_xlabel("Simulation step")
    ax3.set_ylabel("Total modeled cost (USD)")
    ax3.set_title("Red Sea Scenario: Cost by Step")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

recommendation_counts = {}
for result in predictive_results.values():
    for recommendation in result["recommendations"]:
        for word in recommendation.lower().split():
            cleaned_word = word.strip(".,:;()")
            if len(cleaned_word) > 3:
                recommendation_counts[cleaned_word] = (
                    recommendation_counts.get(cleaned_word, 0) + 1
                )
top_terms = sorted(
    recommendation_counts.items(), key=lambda item: item[1], reverse=True
)[:8]
if top_terms:
    terms, counts = zip(*top_terms)
    ax4.bar(terms, counts, color="lightgreen", alpha=0.8)
    ax4.set_xlabel("Terms in threshold-based recommendations")
    ax4.set_ylabel("Frequency")
    ax4.set_title("Recommendation Term Counts")
    ax4.tick_params(axis="x", rotation=40)
    ax4.grid(True, alpha=0.3)
else:
    ax4.text(
        0.5,
        0.5,
        "No recommendation terms available",
        ha="center",
        va="center",
        transform=ax4.transAxes,
    )
    ax4.set_title("Recommendation Term Counts")

plt.tight_layout()
plt.show()

all_recommendations = [
    recommendation
    for result in predictive_results.values()
    for recommendation in result["recommendations"]
]
print("What-if analysis summary")
print(f"  Scenarios analyzed: {len(predictive_results)}")
print(f"  Mean cost-change stabilization indicator: {np.mean(adaptation_speeds):.2f}")
print(f"  Final modeled cost range: ${min(final_costs):,.0f} to ${max(final_costs):,.0f}")
print(f"  Air assignment share range: {min(air_usage):.1f}% to {max(air_usage):.1f}%")
print(f"  Threshold-based recommendations: {len(all_recommendations)}")

## 8. Capabilities, limitations, and appropriate use

### What this simulation demonstrates

- **Connected data flow:** synthetic sensor and market observations pass through a defined processing sequence.
- **Risk-aware decisions:** disruption indicators influence modeled costs and the priority-based assignment policy.
- **Scenario analysis:** users can observe how selected market inputs affect modeled outputs.
- **Performance monitoring:** costs, assignments, alerts, and diagnostic indicators are collected for interpretation.

### Limitations

- This notebook does not ingest live IoT, carrier, port, weather, or market data.
- The risk and delay rules are illustrative and are not validated forecasts.
- The mode-selection policy is a set of ordered rules, not a global optimization algorithm. It does not enforce air capacity or minimize total cost across shipments.
- Fuel price is tracked but is not included in the modeled cost calculation.
- The indicators do not include emissions, service-level trade-offs beyond the stated rules, network interactions among shipments, or uncertainty intervals.
- Repeated stochastic simulation steps are not independent statistical replications, and the notebook does not estimate confidence intervals.

### When the modeling pattern may be useful

This structure can help explain how data ingestion, transformation, risk indicators, decision rules, and reporting fit together in a logistics decision-support workflow. Applying it operationally would require validated data, calibrated assumptions, explicit capacity and service requirements, appropriate governance, and testing against real outcomes. Those additions are outside the scope of this implementation.